# Workshop 2 — Web Scraping multi-pages avec Scrapy
**Story 10 – RegEx et Scraping**

Objectif : reproduire le workshop précédent avec **Scrapy**, un framework dédié au scraping,
en suivant automatiquement la **pagination** (scraping de plusieurs pages).

Réf. brief : *Scraper les données de plusieurs pages avec Scrapy* (A. Warembourg, Medium) et
la documentation officielle *Scrapy Tutorial*.

## Concepts clés de Scrapy
- **Spider** : une classe qui décrit *quoi* scraper et *comment* suivre les liens.
- **`start_urls`** : pages de départ.
- **`parse()`** : méthode appelée pour chaque réponse ; elle `yield` les données extraites.
- **Sélecteurs** : `response.css(...)` ou `response.xpath(...)`.
- **Suivi de liens** : `response.follow(lien, callback=self.parse)` pour la pagination.

Installation (décommenter si besoin) :

In [ ]:
# !pip install scrapy

## Option A — Le spider en fichier autonome
On écrit le spider dans un fichier `.py`. Il se lance ensuite en ligne de commande :

```bash
scrapy runspider books_spider.py -o livres_scrapy.csv
```

La magic `%%writefile` enregistre la cellule dans un fichier.

In [ ]:
%%writefile books_spider.py
import scrapy
import re


class BooksSpider(scrapy.Spider):
    name = 'books'
    start_urls = ['https://books.toscrape.com/catalogue/page-1.html']
    custom_settings = {
        'USER_AGENT': 'Mozilla/5.0 (workshop pedagogique)',
        'DOWNLOAD_DELAY': 0.5,   # politesse : 0,5 s entre les requetes
    }
    NOTES = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

    def parse(self, response):
        for art in response.css('article.product_pod'):
            prix_txt = art.css('p.price_color::text').get('')
            note_mot = art.css('p.star-rating::attr(class)').get('').split()[-1]
            href = art.css('h3 a::attr(href)').get('')
            yield {
                'titre': art.css('h3 a::attr(title)').get(),
                # RegEx dans Scrapy : .re_first() applique un motif directement
                'prix': float(re.search(r'[\d.]+', prix_txt).group()),
                'note': self.NOTES.get(note_mot),
                'id_livre': art.css('h3 a::attr(href)').re_first(r'_(\d+)/'),
                'lien': response.urljoin(href),
            }
        # Pagination : suivre le bouton 'next' tant qu'il existe
        suivant = response.css('li.next a::attr(href)').get()
        if suivant:
            yield response.follow(suivant, callback=self.parse)

## Option B — Lancer Scrapy depuis le notebook
On peut exécuter le spider sans ligne de commande grâce à `CrawlerProcess`.

> Attention : un `CrawlerProcess` ne peut être lancé qu'**une seule fois par noyau (kernel)**.
> Si vous relancez la cellule, redémarrez le kernel avant.

In [ ]:
from scrapy.crawler import CrawlerProcess
from books_spider import BooksSpider

process = CrawlerProcess(settings={
    'FEEDS': {'livres_scrapy.csv': {'format': 'csv', 'overwrite': True}},
    'LOG_LEVEL': 'WARNING',
})
process.crawl(BooksSpider)
process.start()   # bloque jusqu'a la fin du crawl
print('Crawl termine -> livres_scrapy.csv')

## Vérifier le résultat
Scrapy a suivi toutes les pages et exporté un CSV. On le relit avec pandas.

In [ ]:
import pandas as pd
df = pd.read_csv('livres_scrapy.csv')
print('Lignes collectees :', len(df))
df.head()

## Les RegEx dans Scrapy
Scrapy intègre les expressions régulières directement dans les sélecteurs :

- `selector.re(r'motif')` → renvoie **toutes** les correspondances (liste).
- `selector.re_first(r'motif')` → renvoie la **première** correspondance.

Exemple : extraire l'identifiant numérique du lien d'un livre, comme dans le spider ci-dessus :

```python
art.css('h3 a::attr(href)').re_first(r'_(\d+)/')
```

## Conclusion
Scrapy automatise la pagination, la concurrence des requêtes et l'export, avec moins de code
« plomberie » que `requests` + BeautifulSoup. Il est adapté aux **scrapings volumineux et multi-pages**.

| | BeautifulSoup + requests | Scrapy |
|---|---|---|
| Mise en place | rapide | un peu plus longue |
| Multi-pages / volume | manuel | natif et performant |
| RegEx intégrées | via le module `re` | `.re()` / `.re_first()` |

**Cadre légal** : comme pour BeautifulSoup, respecter `robots.txt`, les CGU et un délai entre requêtes
(`DOWNLOAD_DELAY`).